# Notebook 12 — Hybrid Search & Reranking

Combine BM25 keyword search with vector search using Reciprocal Rank Fusion, then rerank with Claude.

In [ ]:
import sys, os, math, re
from collections import Counter
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## BM25 score function

In [ ]:
def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = docs
        self.k1 = k1; self.b = b
        tfs = [Counter(tokenize(d)) for d in docs]
        self.tfs = tfs
        avg_dl = sum(sum(t.values()) for t in tfs) / len(docs)
        self.avg_dl = avg_dl
        n = len(docs)
        self.idf = {}
        for term in {t for tf in tfs for t in tf}:
            df = sum(1 for tf in tfs if term in tf)
            self.idf[term] = math.log((n - df + 0.5) / (df + 0.5) + 1)
    def score(self, query):
        scores = []
        for tf in self.tfs:
            dl = sum(tf.values())
            s = sum(self.idf.get(t, 0) * tf[t] * (self.k1+1) /
                    (tf[t] + self.k1*(1-self.b+self.b*dl/self.avg_dl))
                    for t in tokenize(query))
            scores.append(s)
        return scores


## Reciprocal Rank Fusion

In [ ]:
def rrf(ranked_lists, k=60):
    scores = {}
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked, 1):
            scores[idx] = scores.get(idx, 0) + 1/(k+rank)
    return sorted(scores, key=scores.get, reverse=True)


## Hybrid search demo

In [ ]:
CORPUS = [
    "Python was created by Guido van Rossum in 1991.",
    "Machine learning uses algorithms to learn from data.",
    "Python supports OOP, functional, and imperative styles.",
    "Neural networks are inspired by the human brain.",
    "Guido van Rossum stepped down as benevolent dictator in 2018.",
]

import math
def cosine(a,b):
    dot=sum(x*y for x,y in zip(a,b))
    ma=math.sqrt(sum(x*x for x in a)); mb=math.sqrt(sum(x*x for x in b))
    return dot/(ma*mb) if ma and mb else 0

vocab = sorted({w for c in CORPUS for w in re.findall(r"\b[a-z]+\b", c.lower())})
def embed(text):
    tokens = re.findall(r"\b[a-z]+\b", text.lower())
    tf = Counter(tokens); total = len(tokens) or 1
    return [tf.get(w,0)/total for w in vocab]

vecs = [embed(c) for c in CORPUS]
bm25 = BM25(CORPUS)

def hybrid_search(query, top_k=3):
    q_vec = embed(query)
    dense_scored  = sorted(range(len(CORPUS)), key=lambda i: cosine(q_vec, vecs[i]), reverse=True)
    sparse_scores = bm25.score(query)
    sparse_scored = sorted(range(len(CORPUS)), key=lambda i: sparse_scores[i], reverse=True)
    fused = rrf([dense_scored, sparse_scored])[:top_k]
    return [(CORPUS[i], fused.index(i)) for i in fused]

for q in ["Who created Python?", "neural networks"]:
    print(f"Query: {q!r}")
    for chunk, rank in hybrid_search(q):
        print(f"  [{rank}] {chunk}")
    print()


## LLM Reranker

In [ ]:
import json

def llm_rerank(query, candidates, top_k=2):
    numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(candidates))
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=150, temperature=0,
        messages=[{"role":"user","content":
            f"Rate relevance 0-1 for query: {query!r}\n{numbered}\n"
            f'Reply JSON: [{{"index":0,"score":0.9}},...]. Only JSON.'}]
    )
    try:
        scores = json.loads(r.content[0].text)
        ranked = sorted(scores, key=lambda x: x["score"], reverse=True)
        return [candidates[s["index"]] for s in ranked[:top_k]]
    except Exception:
        return candidates[:top_k]

query = "Who created Python?"
candidates = [c for c, _ in hybrid_search(query, top_k=4)]
reranked = llm_rerank(query, candidates)
print("Reranked top results:")
for c in reranked:
    print(" -", c)


## Exercise
Test the hybrid search with queries that favor BM25 (exact keyword) vs. vector search (paraphrase).

In [ ]:
# Your code here
